In [1]:
!pip install transformers datasets peft sentence-transformers faiss-cpu accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 88.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 39.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━

In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import torch, os

2025-12-06 08:35:08.891987: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765010109.093822      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765010109.156358      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
# =====================================
# Step 3: Load Dataset
# =====================================
# Your dataset should be in JSON format with keys: "prompt" and "completion"
dataset = load_dataset("json", data_files='/kaggle/input/dataset/clean_healthcare_chatbot_15k_expanded_codemixed.jsonl')['train']
print("Sample example:")
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Sample example:
{'prompt': 'Naku stomach pain undi, em cheyyali?', 'completion': 'Stomach pain vunte first light food teesukondi and heavy/spicy/oily food avoid cheyyandi. Warm water or heat pad apply chesi abdominal muscles relax cheyyandi. If vomiting, blood in vomit or severe continuous pain untunte immediate hospital visit avasaram. Mild indigestion ki antacid doctor advice tho teesukovachu but self-medication avoid cheyyandi. Keep hydration and small frequent meals preserve cheyyandi; symptom daily maintain cheyyandi.'}


In [4]:
# =====================================
# Step 4: Load GPT-2 Small and Apply LoRA (Supports Resuming)
# =====================================
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print("🚀 Starting new training...")
model = GPT2LMHeadModel.from_pretrained(model_name)
lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=["c_attn", "c_proj"],
        lora_dropout=0.1,
        bias="none"
)
model = get_peft_model(model, lora_config)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

🚀 Starting new training...


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/layer.py:1803: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [5]:
def tokenize(examples):
    # examples is a dict of lists when batched=True
    texts = [p + c for p, c in zip(examples["prompt"], examples["completion"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=256)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize, batched=True)


Map:   0%|          | 0/10500 [00:00<?, ? examples/s]

In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json

# 1. Load text KB
with open("/kaggle/input/ragdoc/kb_docs_rich.txt", "r", encoding="utf-8") as f:
    kb_texts = [line.strip() for line in f if line.strip()]

# 2. Encode using multilingual model
embed_model = SentenceTransformer("l3cube-pune/indic-sentence-bert-nli")
kb_embeddings = embed_model.encode(kb_texts, convert_to_numpy=True, show_progress_bar=True)

# 3. Build FAISS index
dim = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(kb_embeddings)

# 4. Save everything
faiss.write_index(index, "faiss_index.index")
with open("kb_texts.json", "w", encoding="utf-8") as f:
    json.dump(kb_texts, f, ensure_ascii=False, indent=2)

print(f"✅ FAISS index built with {len(kb_texts)} entries.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ FAISS index built with 63 entries.


In [ ]:
import torch, json, faiss
from sentence_transformers import SentenceTransformer
from transformers import GPT2LMHeadModel, GPT2Tokenizer, pipeline
from peft import PeftModel

# 1☖ Correct path to your model folder, pointing to the actual output directory within the unzipped content
model_dir_adapter = "/kaggle/input/code-mixed-telugu-dataset/output"

# 2☖ Load the base model and tokenizer first
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = GPT2LMHeadModel.from_pretrained(model_name)

# Load the PEFT adapter onto the base model
model = PeftModel.from_pretrained(base_model, model_dir_adapter)

# 3☖ Create text generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")

# 4☖ Load FAISS index and KB
index = faiss.read_index("/kaggle/working/faiss_index.index")
with open("/kaggle/working/kb_texts.json", encoding="utf-8") as f:
    kb_texts = json.load(f)

# 5☖ Load sentence embedding model
embed_model = SentenceTransformer("l3cube-pune/indic-sentence-bert-nli")

# 6☖ Define retrieval function
def retrieve_top_k(query, k=3):
    query_emb = embed_model.encode([query])
    D, I = index.search(query_emb, k)
    return [kb_texts[i] for i in I[0]]

# 7☖ Define RAG response function
def rag_response(user_query):
    context = retrieve_top_k(user_query, k=3)
    prompt = f"### Instruction:\nUser: {user_query}\nContext: {' '.join(context)}\n### Response:\n"
    response = generator(prompt, max_new_tokens=100, temperature=0.7, top_p=0.9)[0]["generated_text"]
    return response.split("### Response:")[-1].strip()

In [ ]:
import json, random

all_data = []
with open("/kaggle/input/dataset/clean_healthcare_chatbot_15k_expanded_codemixed.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # skip empty lines
            try:
                all_data.append(json.loads(line))
            except json.JSONDecodeError:
                print("Skipping invalid line:", line)

# =====================================
# 3️⃣ Split into train/test
# =====================================
random.seed(42)
random.shuffle(all_data)
test_size = 500  # you can choose 500-1000 for evaluation
test_data = all_data[:test_size]
train_data = all_data[test_size:]

In [7]:
# ============================================================
# ⚠ RUN THIS CELL ONCE IN KAGGLE - FULL WORKING IMPLEMENTATION
# ============================================================

import gradio as gr
import faiss, json, numpy as np
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, pipeline,
    AutoModelForSeq2SeqLM
)
import torch


# ==========================================
# 🔧 Load Embedding Model (Retrieval)
# ==========================================
print("🔁 Loading Sentence-BERT Embeddings...")

embed_model = SentenceTransformer(
    "l3cube-pune/indic-sentence-bert-nli",
    device="cuda" if torch.cuda.is_available() else "cpu"
)


# ==========================================
# 📁 Load FAISS + KB
# ==========================================
print("🔍 Loading FAISS Index & Knowledge Base...")

index = faiss.read_index("/kaggle/working/faiss_index.index")

with open("/kaggle/working/kb_texts.json", "r", encoding="utf-8") as f:
    kb_texts = json.load(f)


# ==========================================
# ⚡ Load GPT-2 + Your LoRA Adapter
# ==========================================
print("🧠 Loading GPT2 + LoRA Adapter...")

ADAPTER_PATH = "/kaggle/input/code-mixed-telugu-dataset/output"  # ← correct path

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.load_adapter(ADAPTER_PATH)   # ← main fix

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("🔥 GPT2 Telugu LoRA Loaded Successfully!")


# ==========================================
# 🌐 Load English → Telugu Translation Model
# ==========================================
print("🌍 Loading Translation Model...")

TRANSLATOR_MODEL = "aryaumesh/english-to-telugu"

translator_tokenizer = AutoTokenizer.from_pretrained(TRANSLATOR_MODEL)
translator_model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATOR_MODEL)

SRC_LANG = "en_XX"
TGT_LANG = "te_IN"
translator_tokenizer.src_lang = SRC_LANG


def translate_to_telugu(text):
    try:
        encoded = translator_tokenizer(text, return_tensors="pt")
        generated = translator_model.generate(
            **encoded,
            forced_bos_token_id=translator_tokenizer.lang_code_to_id[TGT_LANG],
            max_length=150, num_beams=5,
            no_repeat_ngram_size=3, repetition_penalty=1.2
        )
        return translator_tokenizer.decode(generated[0], skip_special_tokens=True)
    except:
        return "⚠ Translation Failed."


# ==========================================
# 🧠 RAG Retrieval + GPT2 LoRA Generation
# ==========================================
def retrieve_top_k(query, k=3):
    q_emb = embed_model.encode([query]).astype("float32")
    D, I = index.search(q_emb, k)

    context = []
    for i in I[0]:
        if i != -1 and kb_texts[i] not in context:
            context.append(kb_texts[i])
    return context


def rag_response(user_query, k=3):
    context_list = retrieve_top_k(user_query)
    context = "\n".join(context_list)

    prompt = f"""
You are a Telugu medical assistant. Use knowledge below.

User: {user_query}

Medical Reference:
{context}

Answer in Code-Mixed Telugu (Telugu+English Mix):
"""

    result = generator(prompt, max_new_tokens=250, temperature=0.7, top_p=0.9)[0]["generated_text"]
    final = result.replace(prompt, "").strip()

    return final, context_list


# ==========================================
# 💬 Chat Logic
# ==========================================
def chat_function(message, history):
    response, docs = rag_response(message)

    telugu_translation = translate_to_telugu(response)
    references_te = [translate_to_telugu(c) for c in docs]

    output = f"""
🗣 **Code-Mixed AI Response**  
{response}

🇮🇳 **Telugu (Pure) Translation**  
{telugu_translation}

---

📌 **Retrieved Medical Reference (English)**  
"""
    for x in docs:
        output += f"• {x[:3000]}...\n"

    output += "\n📖 **Retrieved Reference (Telugu Translation)**\n"
    for t in references_te:
        output += f"• {t[:200]}...\n"

    return output


# ==========================================
# 🎨 UI Styling + Launch
# ==========================================
css = """
#chatbot {height: 600px;}
.gradio-container {background:#0d1117;}
.message.user {background:#1f6feb;color:white;border-radius:14px;padding:10px;}
.message.bot {background:#161b22;color:#e6edf3;border:1px solid #30363d;border-radius:14px;padding:10px;}
textarea {background:#161b22;color:white;border-radius:10px;border:1px solid #30363d;}
button {background:#238636;color:white;border-radius:8px;font-weight:bold;}
button:hover {background:#2ea043;}
"""

chatbot = gr.ChatInterface(
    fn=chat_function,
    title="🏥 Aarogyamitra",
    description="Ask medical queries in Telugu-English Mixed.\nExample: 'Naku fever undi em cheyyali?'",
    css=css,
    examples=[
        ["Naku chest pain undi, em cheyyali?"],
        ["Naku fever undi, em cheyyali?"],
        ["Naku cold and cough undi,em cheyali?"]
    ]
)

chatbot.queue().launch(share=True)



🔁 Loading Sentence-BERT Embeddings...
🔍 Loading FAISS Index & Knowledge Base...
🧠 Loading GPT2 + LoRA Adapter...


/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version', 'target_parameters'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(
Device set to use cuda:0


🔥 GPT2 Telugu LoRA Loaded Successfully!
🌍 Loading Translation Model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://ee0507ad6917564dda.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
